# Notebook 05: ChromaDB Vector Store

In this notebook, we will learn how to **store embeddings in a vector database** and search them.

## What You Will Learn

- What ChromaDB is and why we need it
- How to create a vector store
- How to add documents and embeddings
- How to perform similarity search
- How to filter results by metadata

## What is ChromaDB?

**ChromaDB** is an open-source vector database designed for AI applications. It stores:

- **Text** (the original document chunks)
- **Embeddings** (numerical vectors)
- **Metadata** (page numbers, source files, etc.)

When you ask a question, ChromaDB:
1. Converts your question to an embedding
2. Finds the most similar stored embeddings
3. Returns the matching text chunks

## Why Not Just Use Lists?

If you have 10,000 chunks, comparing your question against all of them manually would be slow. ChromaDB uses **approximate nearest neighbor search** to find matches in milliseconds.

## Step 1: Import Required Libraries

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import shutil
import os

# Chroma: LangChain's wrapper for ChromaDB vector store

C:\Users\Ahmed\AppData\Local\Temp\ipykernel_18816\1900899790.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Step 2: Load and Split the PDF

In [2]:
# Load and split the PDF
loader = PyPDFLoader("../data/sample.pdf")
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(pages)

print(f"Total chunks: {len(chunks)}")

Total chunks: 20


## Step 3: Create the Embeddings Model

In [3]:
# Create embeddings model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embeddings model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings model loaded!


## Step 4: Clean Previous Vector Store

If you've run this notebook before, we need to clear the old data.

In [4]:
# Remove old vector store data if it exists
db_path = "../chroma_db"
if os.path.exists(db_path):
    shutil.rmtree(db_path)
    print("Old vector store removed.")
else:
    print("No old vector store found.")

No old vector store found.


## Step 5: Create the Vector Store

In [5]:
# Create a ChromaDB vector store from our documents
# This will:
# 1. Generate embeddings for all chunks
# 2. Store them in ChromaDB
# 3. Save to disk in ../chroma_db/

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=db_path
)

print(f"Vector store created!")
print(f"Number of documents in store: {len(vectorstore.get()['ids'])}")

Vector store created!
Number of documents in store: 20


## Step 6: Perform a Similarity Search

In [6]:
# Search for chunks similar to a question
question = "What is the main topic of this document?"

# similarity_search returns the top-k most similar chunks
results = vectorstore.similarity_search(question, k=3)

print(f"Question: '{question}'")
print(f"\nTop {len(results)} results:\n")

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print()

Question: 'What is the main topic of this document?'

Top 3 results:

--- Result 1 ---
Content: Introduction to Artificial Intelligence
Chapter 4: Natural Language Processing
Natural Language Processing (NLP) is a field of AI that focuses on the interaction between computers and
human language. NLP enables machines to read, understand, and generate human language, making it
possible to build applications like chatbots, translation systems, and text summarizers.
Key NLP tasks include:
- Tokenization: Splitting text into individual words or subwords.
Metadata: {'source': '../data/sample.pdf', 'total_pages': 4, 'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20260729184321', 'page_label': '3', 'page': 2}

--- Result 2 ---
Content: Key NLP tasks include:
- Tokenization: Splitting text into individual words or subwords.
- Sentiment Analysis: Determining whether a piece of text expresses positive, negative, or neutral sentiment.
- Named Entity

## Step 7: Similarity Search with Scores

In [7]:
# similarity_search_with_score returns results with similarity scores
# Higher score = more similar

question = "What is the main topic of this document?"
results_with_scores = vectorstore.similarity_search_with_score(question, k=5)

print(f"Question: '{question}'")
print(f"\nResults with similarity scores:\n")

for i, (doc, score) in enumerate(results_with_scores):
    print(f"Result {i+1} (score: {score:.4f}):")
    print(f"  {doc.page_content[:100]}...")
    print()

Question: 'What is the main topic of this document?'

Results with similarity scores:

Result 1 (score: 1.4556):
  Introduction to Artificial Intelligence
Chapter 4: Natural Language Processing
Natural Language Proc...

Result 2 (score: 1.4690):
  Key NLP tasks include:
- Tokenization: Splitting text into individual words or subwords.
- Sentiment...

Result 3 (score: 1.4918):
  Introduction to Artificial Intelligence
Chapter 3: Deep Learning
Deep Learning is a subset of Machin...

Result 4 (score: 1.6318):
  certain groups.
- Privacy: AI systems often require large amounts of personal data, raising concerns...

Result 5 (score: 1.6481):
  Introduction to Artificial Intelligence
Chapter 1: What is Artificial Intelligence?
Artificial Intel...



## Step 8: Search with Metadata Filtering

In [8]:
# You can filter results by metadata
# For example, search only within specific pages

question = "What is the main topic?"

# Filter to only pages 1-2 (page index 0-1)
results_filtered = vectorstore.similarity_search(
    question,
    k=3,
    filter={"page": {"$lte": 2}}  # Only pages 1 and 2
)

print(f"Filtered results (pages 1-2 only):")
for i, doc in enumerate(results_filtered):
    print(f"  Page {doc.metadata.get('page', 'unknown')}: {doc.page_content[:80]}...")

Filtered results (pages 1-2 only):
  Page 0: structures. Examples include customer segmentation and anomaly detection.
3. Rei...
  Page 0: statistical techniques to find patterns in data.
There are three main types of M...
  Page 2: Introduction to Artificial Intelligence
Chapter 4: Natural Language Processing
N...


## Step 9: Load an Existing Vector Store

In [9]:
# Later, you can reload the saved vector store without re-embedding
vectorstore_loaded = Chroma(
    persist_directory=db_path,
    embedding_function=embeddings
)

# Test that it works
results = vectorstore_loaded.similarity_search("test question", k=1)
print(f"Loaded vector store works! Found {len(results)} results.")

Loaded vector store works! Found 1 results.


## Key Takeaways

1. **ChromaDB** stores embeddings and enables fast similarity search
2. **from_documents()** creates embeddings and stores them automatically
3. **similarity_search()** finds the most relevant chunks for a query
4. **similarity_search_with_score()** also returns similarity scores
5. The vector store can be **persisted** to disk and reloaded later
6. You can **filter** searches by metadata (e.g., page number)

## The RAG Pipeline So Far

```
PDF -> Load -> Split -> Embed -> Store in ChromaDB -> Search
```

## Next Steps

Proceed to **Notebook 06: RAG** to connect retrieval with generation!